# Rain prediction using ERA5 data

Using data from 2000-2023 from ERA5, your task is to predict whether it will rain tomorrow. This is a binary classification task. We define rainy days as days where the precipitation amount > 1 mm.

The data is already preprocessed and in tabular format. We look at five cities, which are the locations of this and last years StuMeTas.

More information about the data can be found here: [ERA5 hourly data on single levels from 1940 to present
](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=overview$0)


First, we install and import relevant packages.

In [ ]:
!pip -q install cdsapi xarray netCDF4 scikit-learn tensorflow openmeteo-requests requests-cache retry-requests

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
from sklearn.preprocessing import StandardScaler

### Load/request data

Historical data: ```rain_prediction_historical.csv```

Data for training: ```rain_prediction.csv```


In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

def fetch_weather(lat, lon, name, start="2000-01-01", end="2023-12-31"):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start, "end_date": end,
        "daily": [
            "temperature_2m_max", "temperature_2m_min",
            "precipitation_sum", "wind_speed_10m_max",
            "pressure_msl_mean", "cloud_cover_mean",
            "relative_humidity_2m_mean", "et0_fao_evapotranspiration"
        ],
        "timezone": "UTC"
    }
    response = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)[0]
    daily = response.Daily()
    # for historical data: Only tmax, tmin, precip
    df = pd.DataFrame({
        "date":        pd.date_range(start=pd.to_datetime(daily.Time(), unit="s"), periods=daily.Variables(0).ValuesAsNumpy().shape[0], freq="D"),
        "tmax":        daily.Variables(0).ValuesAsNumpy(),
        "tmin":        daily.Variables(1).ValuesAsNumpy(),
        "precip":      daily.Variables(2).ValuesAsNumpy(),
        "wind_speed":  daily.Variables(3).ValuesAsNumpy(),
        "pressure":    daily.Variables(4).ValuesAsNumpy(),
        "cloud_cover": daily.Variables(5).ValuesAsNumpy(),
        "humidity":    daily.Variables(6).ValuesAsNumpy(),
        "et0":         daily.Variables(7).ValuesAsNumpy(),
    })
    df["location"] = name
    return df

locations = {
    "Hamburg": (53.55, 10.00),
    "Karlsruhe": (49.10, 8.24),
    "Leipzig": (51.20, 12.22),
    "Innsbruck": (47.16, 11.24),
    "Berlin": (52.31, 13.24),
}

all_data = pd.concat([fetch_weather(lat, lon, name) for name, (lat, lon) in locations.items()])
all_data.to_csv("rain_prediction.csv", index=False)

In [ ]:
# Load data from files
historical_data = pd.read_csv("rain_prediction_historical.csv", parse_dates=["date"])
data            = pd.read_csv("rain_prediction.csv", parse_dates=["date"])

## Inspect historical data

To start, we look at the historical data and compare this with statistical values (see below for the different cities, Karlsruhe is not available) to validate our data. For the comparison, it is sufficient to look at only one or two of the cities.

* [Hamburg](https://worldweather.wmo.int/en/city.html?cityId=55$0)

* [Leipzig](https://worldweather.wmo.int/en/city.html?cityId=1350$0)

* [Innsbruck](https://worldweather.wmo.int/en/city.html?cityId=905$0)

* [Berlin](https://worldweather.wmo.int/en/city.html?cityId=59$0)

In [ ]:
print(historical_data.head())
print(historical_data.groupby("location")["precip"].describe())

In [ ]:
# How often does it rain in each city?
historical_data["rain_today"] = (historical_data["precip"] > 1.0).astype(int)
historical_data.groupby("location")["rain_today"].mean().sort_values().plot(kind="barh")
plt.title("Fraction of rainy days by location")
plt.xlabel("Fraction of days with >1mm precipitation")
plt.show()


In [ ]:
# Select the data for one city
historical_data_hamburg = historical_data[historical_data["location"] == 'Hamburg']

In [ ]:
historical_data_hamburg

In [ ]:
# Plot temperature over time
plt.figure(figsize=(12, 3))
plt.plot(historical_data_hamburg['date'], historical_data_hamburg['tmax'], linewidth=0.5, color='tomato', label="max. temp.")
plt.plot(historical_data_hamburg['date'], historical_data_hamburg['tmin'], linewidth=0.5, label="min. temp.")
plt.legend(loc="best")
plt.title('Daily Temperatures in Hamburg (1971-2000)')
plt.ylabel('°C')
plt.show()



In [ ]:
# Mean monthly rainfall
monthly_total_rainfall = (
    historical_data_hamburg.groupby([
        historical_data_hamburg["date"].dt.year,
        historical_data_hamburg["date"].dt.month
    ])["precip"]
    .sum()
)
mean_monthly_rainfall = monthly_total_rainfall.groupby(level=1).mean()
print(mean_monthly_rainfall)

In [ ]:
# Mean number of rainy days
rainy_days = (historical_data_hamburg["precip"] > 1.0).astype(int)

monthly_rainy_days = (
    rainy_days.groupby([
        historical_data_hamburg["date"].dt.year,
        historical_data_hamburg["date"].dt.month
    ])
    .sum()
)
mean_rainy_days = monthly_rainy_days.groupby(level=1).mean()
print(mean_rainy_days)

In [ ]:
# Mean daily minimum temperature
monthly_mean_daily_min_temp = (
    historical_data_hamburg.groupby(historical_data_hamburg["date"].dt.month)["tmin"]
    .mean()
)
print(monthly_mean_daily_min_temp)

In [ ]:
# Mean daily maximum temperature
monthly_mean_daily_max_temp = (
    historical_data_hamburg.groupby(historical_data_hamburg["date"].dt.month)["tmax"]
    .mean()
)
print(monthly_mean_daily_max_temp)



---



## Data preparation

Now, we start to work with the data for training the ML models, ```rain_prediction.csv```. First, we create the labels. For this, we create a new column ```rain_today``` that contains an identifier for rain or no rain (1: precip > 1 mm, 0: precip $\leq$ 1 mm). Then, we obtain the label by creating a second column ```rain_tomorrow``` by shifting the content of ```rain_today``` by one day.



In [ ]:
# Plot temperature over time (just for visualization)
plt.figure(figsize=(12, 3))
plt.plot(data[data["location"] == 'Hamburg']['date'], data[data["location"] == 'Hamburg']['tmax'], linewidth=0.5, color='tomato', label="max. temp.")
plt.plot(data[data["location"] == 'Hamburg']['date'], data[data["location"] == 'Hamburg']['tmin'], linewidth=0.5, label="min. temp.")
plt.legend(loc="best")
plt.title('Daily Temperatures in Hamburg (2000-2023)')
plt.ylabel('°C')
plt.show()



In [ ]:
# Create column for rain/no rain
data["rain_today"] = (data["precip"] > 1.0).astype(int)
data

In [ ]:
data = data.sort_values(["location", "date"])

# Create label
data["rain_tomorrow"] = data.groupby("location")["rain_today"].shift(-1)

In [ ]:
# Add date information
data["year"] = data["date"].dt.year
data["month"] = data["date"].dt.month
data["day"] = data["date"].dt.dayofyear
data

### Data splitting into training/validation/test sets

In [ ]:
# Split by time to avoid temporal autocorrelation
# Select the years 2000-2015 for training, 2016-2019 for validation and 2020-2023 for testing
data_train = data[(data["year"] < 2016)]
data_val = data[(data["year"] >= 2016) & (data["year"] <= 2019)]
data_test = data[(data["year"] >= 2020)]
print(f"NaNs: \nTraining: \n{data_train.isna().sum()} \nValidation: \n{data_val.isna().sum()} \nTest: \n{data_test.isna().sum()}")
data_train = data_train.dropna()
data_val = data_val.dropna()
data_test = data_test.dropna()

In [ ]:
# Split features and labels for all datasets
features = [
    "tmax", "tmin", "precip", "wind_speed", "pressure", "cloud_cover", "humidity",
    "et0", "rain_today", "month", "year"
]
X_train, X_val, X_test = data_train[features], data_val[features], data_test[features]
y_train, y_val, y_test = data_train["rain_tomorrow"], data_val["rain_tomorrow"], data_test["rain_tomorrow"]

In [ ]:
# We use seaborn to inspect the training data
sns.pairplot(data_train[features[:-3]])

### Feature scaling

Next, we scale the features. There are different ways to do this. Here, we use **standard scaling**, which means that we standardize thefeatures by removing the mean and scaling to unit variance.
* [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html$0)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

## Training

Now, we look at different models for prediction. For training, we use the scaled training data.

### Evaluation

We create a custom evaluation function which we use to evaluate the models. At the moment, we only consider two metrics, **accuracy** and the **$F_1$ score**, and look at the classification report. Do a quick search what those metrics tell us. You can add also other metrics if you like!
* [Accuracy score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html#sklearn.metrics.accuracy_score$0)
* [F1 score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html#sklearn.metrics.f1_score$0)
* [Classification report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html#sklearn.metrics.classification_report$0)
* [Classification metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics$0)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
def evaluate(model_instance, y, y_pred, dataset_name, model_name):
  accuracy = accuracy_score(y, y_pred)
  f1 = f1_score(y, y_pred)
  report = classification_report(y, y_pred)
  print("=" * 60)
  print(f" {model_name}, {dataset_name}")
  print("=" * 60)
  print(f" Accuracy: {accuracy:.3f} \n F1 score: {f1:.3f} \n Classification report: \n {report} \n")



---



### **Logistic regression baseline**

Before testing out ML architectures, we start with a simple logistic regression baseline, which is essential to every ML workflow. More information:
* [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html$0)
* [Wikipedia](https://en.wikipedia.org/wiki/Logistic_regression#Supervised_machine_learning$0)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# create model instance
logistic_regression = LogisticRegression()

In [ ]:
# fit the model
logistic_regression.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions
y_pred_train = logistic_regression.predict(X_train_scaled)
y_pred_val = logistic_regression.predict(X_val_scaled)

In [ ]:
# Compute metrics using our custom function
evaluate(
    model_instance = logistic_regression,
    y = y_train,
    y_pred = y_pred_train,
    dataset_name = "Training",
    model_name = "Logistic regression"
)
evaluate(
    model_instance = logistic_regression,
    y = y_val,
    y_pred = y_pred_val,
    dataset_name = "Validation",
    model_name = "Logistic regression"
)



---



### **Decision trees**

As you can see, logistic regression already is a good start. However, the relationships we are trying to model are often not linear.

Next, we look at decision trees. The steps are the same as for logistic regression. We start with a very shallow tree.
* [Decision trees](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.score$0)

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

In [ ]:
decision_tree = DecisionTreeClassifier(max_depth=3) # shallow tree

In [ ]:
decision_tree.fit(X_train_scaled, y_train)

Let's first have a look at the tree. You can find more information here:
* [Plot tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html$0)
* [Understanding the decision tree structure](https://scikit-learn.org/stable/auto_examples/tree/plot_unveil_tree_structure.html$0)
* [Gini impurity](https://en.wikipedia.org/wiki/Decision_tree_learning#Gini_impurity$0)

In [ ]:
fig, ax = plt.subplots(figsize=(25, 10))
plot_tree(decision_tree, filled=True, ax=ax, fontsize=12, feature_names=X_train.columns, class_names=["no_rain", "rain"]);

Look at the tree in detail. Does it match your expectations?

### Compute a full tree

Our first tree was very shallow. To make predictions, we compute a full tree.

In [ ]:
decision_tree = DecisionTreeClassifier() # full tree

In [ ]:
decision_tree.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions
y_pred_train = decision_tree.predict(X_train_scaled)
y_pred_val = decision_tree.predict(X_val_scaled)

In [ ]:
# Compute metrics using our custom function
evaluate(
    model_instance = decision_tree,
    y = y_train,
    y_pred = y_pred_train,
    dataset_name = "Training",
    model_name = "Decision tree"
)
evaluate(
    model_instance = decision_tree,
    y = y_val,
    y_pred = y_pred_val,
    dataset_name = "Validation",
    model_name = "Decision tree"
)

Do you notice something here?

You can see that we have perfect scores on the training set, but the scores on the validation set are not so good. This is a classic example of **overfitting**.

But this is to be expected. We built our tree to perfectly fit the training set, so that only one sample remains in the final leafs.

Therefore, we need to adjust the tree to prevent overfitting. This is a simple form of **hyperparameter tuning**.

Here, we reduce the complexity of the tree by increasing the number of samples in each leaf and setting the maximum depth of the tree. You can experiment here!

In [ ]:
decision_tree = DecisionTreeClassifier(min_samples_leaf = 10, max_depth=5)
decision_tree.fit(X_train_scaled, y_train)
y_pred_train = decision_tree.predict(X_train_scaled)
y_pred_val = decision_tree.predict(X_val_scaled)
evaluate(
    model_instance = decision_tree,
    y = y_train,
    y_pred = y_pred_train,
    dataset_name = "Training",
    model_name = "Decision tree"
)
evaluate(
    model_instance = decision_tree,
    y = y_val,
    y_pred = y_pred_val,
    dataset_name = "Validation",
    model_name = "Decision tree"
)



---



### **Random forests**

Going one step further, we will now implement a random forest. The idea behind random forests is to have many individual decision trees that are all a little different. At each split, only a random subset of features and samples is taken into account for deciding the best split. Because the errors of the individual trees will be independent from each other, averaging the prediction will reduce the error.
* [Random Forest Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html$0)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
random_forest = RandomForestClassifier(n_estimators=100, random_state=42) # random forest

In [ ]:
random_forest.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions
y_pred_train = random_forest.predict(X_train_scaled)
y_pred_val = random_forest.predict(X_val_scaled)

In [ ]:
# Compute metrics using our custom function
evaluate(
    model_instance = random_forest,
    y = y_train,
    y_pred = y_pred_train,
    dataset_name = "Training",
    model_name = "Random forest"
)
evaluate(
    model_instance = random_forest,
    y = y_val,
    y_pred = y_pred_val,
    dataset_name = "Validation",
    model_name = "Random forest"
)

Again, we see overfitting and have to tune the hyperparameters. Try this out yourself! If you want, you can also try to implement a random search that automatically tests different combinations of hyperparameters.

In [ ]:
random_forest = RandomForestClassifier(n_estimators=100, n_jobs=-1, min_samples_leaf=20)
random_forest.fit(X_train_scaled, y_train)
y_pred_train = random_forest.predict(X_train_scaled)
y_pred_val = random_forest.predict(X_val_scaled)
evaluate(
    model_instance = random_forest,
    y = y_train,
    y_pred = y_pred_train,
    dataset_name = "Training",
    model_name = "Random forest"
)
evaluate(
    model_instance = random_forest,
    y = y_val,
    y_pred = y_pred_val,
    dataset_name = "Validation",
    model_name = "Random forest"
)

#### Confusion matrix

To get a better understanding of model performance, we can also look at the confusion matrix:
* [ConfusionMatrixDisplay](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html$0)


In [ ]:
ConfusionMatrixDisplay.from_estimator(random_forest, X_train_scaled, y_train, display_labels=["No rain", "Rain"])
plt.title("Random Forest, training")
plt.show()

In [ ]:
ConfusionMatrixDisplay.from_estimator(random_forest, X_val_scaled, y_val, display_labels=["No rain", "Rain"])
plt.title("Random Forest, validation")
plt.show()

#### Feature importances

We do not only want a prediction that yields high metric scores. Ideally, we also want to understand **why** we are getting the prediction.

One way to do this is to look at feature importance, which tells us about the importance of individual features for the final prediciton. Two common methods to identify important features are permutation importance and the Gini importance.

In permutation importance, each column of the dataset is randomly shuffled. Then, we check by how much the prediction skill decreases. This is simply a measure of how important each input feature is for making a good prediction. In other words, without this feature how much worse would the prediction get?

Gini importance calculates each features importance as the sum over the number of splits (across all tress) that include the feature, proportionally to the number of samples it splits.

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
permutation_importance(random_forest, X_train_scaled, y_train, n_repeats=1, n_jobs=-1)

In [ ]:
random_forest.feature_importances_

In [ ]:
# Create a dataframe for plotting
importances_df = pd.DataFrame(
    data={'Feature': X_train.columns, 'Feature importance': random_forest.feature_importances_},
    columns = ['Feature', 'Feature importance']
)
importances_df.sort_values('Feature importance', inplace=True, ascending=False)

In [ ]:
sns.barplot(data=importances_df, x='Feature importance', y='Feature')
plt.xscale('log')

What do the results tell you? Are they what you expected?

#### **Evaluation per city**

Finally, after tuning the hyperparameters of our random forest model, we want to evaluate how the model performs for each city, using the test set.

In [ ]:
data_test["prediction"] = random_forest.predict(X_test_scaled)
# data_test = data_test.dropna()

for city, group in data_test.groupby("location"):
  f1 = f1_score(group["rain_tomorrow"], group["prediction"])
  accuracy = accuracy_score(group["rain_tomorrow"], group["prediction"])
  print(f"{city:20s} Accuracy: {accuracy:.2f}, F1 score: {f1:.2f}")


### **Neural network**

Finally, we want to train a neural network. For this, we use the **PyTorch** library.

* [PyTorch Tutorial and documentation](https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html$0)

In [ ]:
import torch
import torch.nn as nn

First, we create the model class ```RainNN```. Then, we use ```torch.tensor``` for our datasets, so that they are in the correct format for the NN model class. Finally, we implement the training loop. As the loss function, we use the ```BCELoss``` and the Adam optimizer.
* [BCELoss](https://docs.pytorch.org/docs/2.12/generated/torch.nn.BCELoss.html$0)
* [Pytorch optimizer information](https://docs.pytorch.org/docs/2.12/optim.html$0)
* [Adam optimizer](https://docs.pytorch.org/docs/2.12/generated/torch.optim.Adam.html#torch.optim.Adam$0)

In [ ]:
# model class
class RainNN(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.NN = nn.Sequential(
        nn.Linear(n_features, 64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Sigmoid()
    )
  def forward(self, x):
    return self.NN(x).squeeze()

In [ ]:
# PyTorch tensors
X_train_torch = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_torch = torch.tensor(y_train.to_numpy(), dtype=torch.float32)

In [ ]:

model = RainNN(X_train_torch.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

for epoch in range(100): # we train for 100 epochs
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train_torch), y_train_torch)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/100 — Loss: {loss.item():.4f}")

If you still have time, you could try to visualize the learning curve. This is very helpful for detecting overfitting!

Finally, we make predictions. For this, we again have to create tensors. Then, we have to set the model to evaluation mode, this disables dropout, which is only active during training. Our model ends with a sigmoid, so it already outputs a probability between 0 and 1. Bacuse of this, we convert the probabilities to classes at the end. ```torch.no_grad()``` is important. Without it, PyTorch still tracks gradients, which wastes memory and time.

Here, we are making predictions on all datasets. If you still have enough time, you could do some hyperparameter tuning first.

In [ ]:
# PyTorch tensors
X_val_torch = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_torch = torch.tensor(y_val.to_numpy(), dtype=torch.float32)
X_test_torch = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_torch = torch.tensor(y_test.to_numpy(), dtype=torch.float32)

# We create a small prediction function
def predict_nn(model, X, y):
  model.eval()
  with torch.no_grad():
    y_prob = model(X).numpy()  # predicted probabilities

  # Convert probabilities to binary labels using 0.5 as the threshold
  y_pred = (y_prob >= 0.5).astype(int)
  return y_prob, y_pred

y_prob_train, y_pred_train = predict_nn(model=model, X=X_train_torch, y=y_train_torch)
y_prob_val, y_pred_val = predict_nn(model=model, X=X_val_torch, y=y_val_torch)
y_prob_test, y_pred_test = predict_nn(model=model, X=X_test_torch, y=y_test_torch)

In [ ]:
# Compute metrics using our custom function
evaluate(
    model_instance = model,
    y = y_train,
    y_pred = y_pred_train,
    dataset_name = "Training",
    model_name = "Neural network"
)
evaluate(
    model_instance = model,
    y = y_val,
    y_pred = y_pred_val,
    dataset_name = "Validation",
    model_name = "Neural network"
)
evaluate(
    model_instance = model,
    y = y_test,
    y_pred = y_pred_test,
    dataset_name = "Test",
    model_name = "Neural network"
)

#### Inspect the probabilites

One advantage of a neural network over a simple threshold classifier is that the probabilities are more informative than simple classes

In [ ]:
data_test_final = X_test.copy()
data_test_final["observed"] = y_test.values
data_test_final["probability"] = y_prob_test
data_test_final["predicted"] = y_pred_test

# Plot predicted probability for a single city
city_mask = data_test["location"].values == "Hamburg"

plt.figure(figsize=(12, 3))
plt.plot(data_test_final.loc[city_mask, "probability"], color="steelblue", linewidth=0.8, label="P(rain)")
plt.scatter(
    data_test_final.index[city_mask & (data_test_final["observed"] == 1)],
    data_test_final.loc[city_mask & (data_test_final["observed"] == 1), "probability"],
    color="tomato", s=5, label="Observed rain", zorder=3
)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="Decision threshold")
plt.legend()
plt.title("Predicted probability of rain: Hamburg (test dataset)")
plt.ylabel("P (rain tomorrow)")
plt.ylim(0, 1)
plt.show()


#### Make a prediction for a new day

You can experiment here! Maybe you can find out the values for today or yesterday and see what the model predicts?

In [ ]:
new_day = pd.DataFrame([{
    "tmax": 22.0,
    "tmin": 15.0,
    "tmean": 18.0,
    "precip": 0.0,   # dry today
    "wind_speed": 8.0,
    "pressure": 1005.0,  # slightly low
    "cloud_cover": 0.8,   # very cloudy
    "humidity": 85.0,
    "et0": 0.55,
    "rain_today": 1,
    "month": 5,
    "year": 2026,
}])

new_day_scaled = scaler.transform(new_day[features])
new_tensor = torch.tensor(new_day_scaled, dtype=torch.float32)

model.eval()
with torch.no_grad():
    prob = model(new_tensor).item()

print(f"Predicted probability of rain tomorrow: {prob:.1%}")
print(f"Decision: {'Rain expected' if prob >= 0.5 else 'No rain expected'}")
